In [ ]:
# LLM tag: "qwen2.5-7b", "qwen3-8b", "llama3.1-8b", "nemotron-nano-8b", "gemini-2.5-flash", or a Hugging Face id.
LLM = "qwen2.5-7b"
# Tasks: ["enrich"], ["classify"] or ["enrich", "classify"].
TASK = ["enrich", "classify"]
# True: 50 test pairs and their terms only, to check the pipeline. False: the full run.
SMOKE = False
# Enrichment scope: "all" (train, val and test terms) or "test" (test terms only).
SCOPE = "all"
# Classifier inputs: any of "plain", "codraft", "own", "expanded", "category".
VARIANTS = ["plain", "codraft", "own"]
# Few-shot sizes: 0 or a multiple of 5 (two demonstrations per label), e.g. [0, 10].
SHOTS = [0, 10]
# vLLM to install: a pip spec such as "vllm==0.30.0".
VLLM_PIP = "vllm==0.30.0"
# None, or a folder holding an earlier zip of the same LLM (e.g. "/kaggle/input") to resume from.
RESUME_FROM = None

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/CoDraft-Bench.git"
BRANCH = "new"
DATA_ROOT = "/kaggle/input/datasets/nhtquyn/codraft"


def sh(*cmd: str, **kw: object) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, **kw)


if os.path.isdir("/kaggle/working"):
    OUT_ROOT = "/kaggle/working"
    REPO_DIR = f"{OUT_ROOT}/CoDraft-Bench"
    if os.path.isdir(REPO_DIR):
        sh("git", "-C", REPO_DIR, "pull", "--ff-only")
    else:
        sh("git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR)
    sh(sys.executable, "-m", "pip", "install", "-q", VLLM_PIP)
    sh(sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio")
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-llm.txt")
    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
        for key in ("HF_TOKEN", "GEMINI_API_KEY"):
            try:
                os.environ[key] = secrets.get_secret(key)
                print(f"Secret {key}: set")
            except Exception:
                print(f"Secret {key}: not set")
    except ImportError:
        pass
else:
    REPO_DIR = os.getcwd()
    while REPO_DIR != "/" and not os.path.isfile(os.path.join(REPO_DIR, "llm_runner.py")):
        REPO_DIR = os.path.dirname(REPO_DIR)
    OUT_ROOT, DATA_ROOT = REPO_DIR, None

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
commit = subprocess.run(
    ["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True
).stdout.strip()
branch = subprocess.run(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True
).stdout.strip()
print(f"Repo {REPO_DIR} | branch {branch} | commit {commit}")
assert branch == BRANCH, f"Expected branch {BRANCH}, got {branch}"

In [ ]:
if DATA_ROOT and not os.path.isdir(DATA_ROOT):
    DATA_ROOT = None
tasks = [TASK] if isinstance(TASK, str) else list(TASK)
cmd = [sys.executable, "-u", "llm_runner.py", "--llm", LLM, "--task", *tasks, "--scope", SCOPE]
cmd += ["--variants", *VARIANTS, "--shots", *map(str, SHOTS), "--out-root", OUT_ROOT]
if DATA_ROOT:
    cmd += ["--data-root", DATA_ROOT]
if SMOKE:
    cmd.append("--smoke")
if RESUME_FROM:
    cmd += ["--resume-from", RESUME_FROM]
sh(sys.executable, "llm_runner.py", "--list")
sh(*cmd, env={**os.environ, "PYTHONUNBUFFERED": "1"})

In [ ]:
import glob
import json

from config.llms import safe_tag

prefix = "SMOKE_" if SMOKE else ""
tag = prefix + safe_tag(LLM)
meta = f"{OUT_ROOT}/attributes/{tag}_enrich.json"
if os.path.isfile(meta):
    with open(meta) as fh:
        r = json.load(fh)
    print(
        f"Enrich {r['tag']}: {r['n_done']}/{r['n_terms']} terms, {r['n_failed']} failed, "
        f"by mode {r['by_mode']}, generic natures {r['generic_nature']}, "
        f"{r['completion_tokens'] / 1e6:.2f} M output tokens, {r['seconds'] / 60:.1f} min"
    )
    print(f"  Revision {r['revision']} | variant splits written: {r['variant_splits']}")

for f in sorted(glob.glob(f"{OUT_ROOT}/results/{prefix}llm_{safe_tag(LLM)}_*_metrics.json")):
    with open(f) as fh:
        r = json.load(fh)
    print(
        f"\n{r['tag']}: Macro-F1 {r['f1_macro'] * 100:.2f} | QWK {r['qwk']:.4f} | MAE {r['mae']:.4f} | "
        f"n={r['n_test']} | unanswered {r['settings']['n_failed']}"
    )
    print(
        "  Per-class F1: " + " / ".join(f"{c['name']} {c['f1'] * 100:.2f}" for c in r["per_class"])
    )

zips = sorted(glob.glob(f"{OUT_ROOT}/*codraft_llm_*.zip"), key=os.path.getmtime)
if zips:
    print(f"\nDownload this file: {zips[-1]} ({os.path.getsize(zips[-1]) / 2**20:.1f} MB)")
else:
    print("\nNo zip found; the run cell may have failed before packaging.")